In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import VotingClassifier, GradientBoostingClassifier, RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE
from catboost import CatBoostClassifier
from sklearn.calibration import CalibratedClassifierCV
import pickle



# ---------------------------
# 1. Load data
# ---------------------------
df = pd.read_csv("../data/heart_2020_cleaned.csv")
df['HeartDisease'] = df['HeartDisease'].map({'No':0, 'Yes':1})

X = df.drop("HeartDisease", axis=1)
y = df["HeartDisease"]

# ---------------------------
# 2. Column types
# ---------------------------
categorical_cols = X.select_dtypes(include=['object']).columns
numeric_cols = X.select_dtypes(include=['int64','float64']).columns

# ---------------------------
# 3. Preprocessing
# ---------------------------
preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols),
    ('num', StandardScaler(), numeric_cols)
])

# ---------------------------
# 4. Train-test split
# ---------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

pos = sum(y_train == 1)
neg = sum(y_train == 0)
ratio = neg / pos

# ---------------------------
# 5. Handle imbalance with SMOTE
# ---------------------------
#smote = SMOTE(random_state=42)

# ---------------------------
# 6. Define models
# ---------------------------
# Random Forest with class weight
rf = RandomForestClassifier(
    n_estimators=600,
    max_depth=8,
    min_samples_split=5,
    min_samples_leaf=2,
    max_features="sqrt",
    class_weight="balanced_subsample",
    bootstrap=True,
    random_state=42
)
rf_cal = CalibratedClassifierCV(rf, method='sigmoid', cv=3)

# Gradient Boosting
gb = GradientBoostingClassifier(
    n_estimators=400,
    learning_rate=0.05,
    max_depth=3,
    subsample=0.7,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42
)
gb_cal = CalibratedClassifierCV(gb, method='sigmoid', cv=3)

# XGBoost with scale_pos_weight
xgb = XGBClassifier(
    n_estimators=400,
    max_depth=4,
    learning_rate=0.08,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=ratio,
    objective='binary:logistic',
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)
xgb_cal = CalibratedClassifierCV(xgb, method='sigmoid', cv=3)

# Logistic Regression with class weight
lr = LogisticRegression(
    C=0.5,
    penalty="l2",
    solver="liblinear",
    class_weight="balanced",
    max_iter=1000,
    random_state=42
)

cat = CatBoostClassifier(
    iterations=600,
    depth=6,
    learning_rate=0.03,
    loss_function="Logloss",
    eval_metric="Recall",
    l2_leaf_reg=3,
    random_strength=1.0,
    bagging_temperature=0.2,
    border_count=128,
    grow_policy="SymmetricTree",
    class_weights=[1, 8],   # adjust imbalance (8x minority boost)
    verbose=0,
    random_state=42
)


# ---------------------------
# 7. Voting Classifier
# ---------------------------
voting_clf = VotingClassifier(
    estimators=[('rf', rf_cal), ('gb', gb_cal), ('xgb', xgb_cal), ('lr', lr), ('cat', cat)],
    voting='soft',  # use probabilities
    weights=[2, 1, 2, 1, 3], # XGB + RF contribute more
    n_jobs=-1
)

# ---------------------------
# 8. Full pipeline
# ---------------------------
pipeline = Pipeline([
    ('preprocess', preprocessor),
    #('smote', smote),
    ('voting', voting_clf)
])

# ---------------------------
# 8. Train
# ---------------------------
pipeline.fit(X_train, y_train)

with open("trained_pipe_voting.sav", "wb") as f:
    pickle.dump(pipeline, f)

# ---------------------------
# 9. Predict probabilities and classes
# ---------------------------
y_proba = pipeline.predict_proba(X_test)[:,1]  # risk score (0-1)
threshold = 0.19
y_pred = (y_proba >= threshold).astype(int)

# ---------------------------
# 10. Evaluate
# ---------------------------
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))


In [ ]:
import pickle
pickle.dump(pipeline,
            open(file='../models/trained_pipe_soft_voting.sav',
                 mode='wb'))

In [ ]:
background_sample = X_train.sample(n=50, random_state=42)
with open('../models/background_sample.sav', 'wb') as f:
    pickle.dump(background_sample, f)